In [3]:
import pandas as pd
import numpy as np
import os

# 1. Chargement du dataset brut
raw_path = 'raw_data/dmi_weather_raw.csv'
if not os.path.exists(raw_path):
    raw_path = 'dmi_weather_raw.csv'

print("⏳ Chargement des données brutes pour audit global...")
df_raw = pd.read_csv(raw_path)

# Conversion impérative en datetime
df_raw['HourUTC'] = pd.to_datetime(df_raw['HourUTC'])

print(f"✅ Dataset chargé : {df_raw.shape[0]} lignes de relevés élémentaires.")

# ==========================================================
# 1. ANALYSE DE LA GRANULARITÉ ET DU PAS DE TEMPS
# ==========================================================
print("\n" + "="*50)
print("1. AUDIT DE LA GRANULARITÉ TEMPORELLE")
print("="*50)

for zone in df_raw['PriceArea'].unique():
    df_zone = df_raw[df_raw['PriceArea'] == zone].sort_values('HourUTC')
    
    # Calcul de la différence de temps entre chaque ligne successive
    time_deltas = df_zone['HourUTC'].diff().value_counts()
    
    print(f"\n[Zone {zone}] Fréquence des intervalles de temps détectés :")
    for delta, count in time_deltas.items():
        print(f"  - Intervalle : {delta} -> {count} fois")

# ==========================================================
# 2. ALIGNEMENT SUR UNE TIMELINE THÉORIQUE PARFAITE
# ==========================================================
# On définit les bornes exactes de notre horizon d'étude (2022-2024)
start_theorique = pd.to_datetime('2022-01-01 00:00:00')
end_theorique = pd.to_datetime('2024-12-31 23:50:00') # au pas 10-min

print("\n" + "="*50)
print("2. RECONSTRUCTION ET COMPTAGE DES NaN DIRECTS & SQUELETTIQUES")
print("="*50)

# D'après la granularité observée (10 minutes), on génère la timeline théorique de référence
perfect_timeline = pd.date_range(start=start_theorique, end=end_theorique, freq='10min')
hours_theoretical = len(perfect_timeline)
print(f"Plage théorique attendue : {start_theorique} à {end_theorique}")
print(f"Nombre d'horodatages (timestamps) attendus par zone : {hours_theoretical}")

# Dictionnaire pour stocker nos datasets alignés pour l'analyse des trous
df_aligned_zones = {}

for zone in ['DK1', 'DK2']:
    df_zone = df_raw[df_raw['PriceArea'] == zone].copy()
    
    # Nettoyage des doublons stricts de timestamps s'il y en a
    duplicates = df_zone.duplicated(subset=['HourUTC']).sum()
    if duplicates > 0:
        print(f"⚠️ Alerte : {duplicates} doublons de timestamps trouvés dans la zone {zone}. Suppression.")
        df_zone = df_zone.drop_duplicates(subset=['HourUTC'])
        
    # Reindexation sur la timeline parfaite pour faire apparaître les lignes physiquement manquantes (trous de réseau)
    df_zone = df_zone.set_index('HourUTC').reindex(perfect_timeline).reset_index()
    df_zone = df_zone.rename(columns={'index': 'HourUTC'})
    
    # Stockage
    df_aligned_zones[zone] = df_zone
    
    print(f"\n🔍 Statut des données manquantes pour la zone {zone} (après alignement) :")
    for col in ['temp_dry', 'wind_speed', 'wind_dir']:
        total_nans = df_zone[col].isnull().sum()
        pct_nans = (total_nans / hours_theoretical) * 100
        print(f"  - {col} : {total_nans} valeurs manquantes ({pct_nans:.2f}%)")

# ==========================================================
# 3. ANALYSE DE LA LONGUEUR MAXIMALE ET DE LA DYNAMIQUE DES TROUS (GAPS)
# ==========================================================
print("\n" + "="*50)
print("3. ANALYSE DE LA DISTRIBUTION DES GAPS (CONSECUTIVE NaNs)")
print("="*50)

def analyze_gap_lengths(series):
    """
    Calcule la longueur de chaque bloc de NaNs consécutifs dans une série.
    """
    is_nan = series.isnull()
    # Identification des blocs consécutifs
    gap_ids = (~is_nan).cumsum()
    # Compte la taille de chaque bloc de NaNs
    gap_lengths = is_nan.groupby(gap_ids).sum()
    # On ne garde que les vrais gaps (longueur > 0)
    return gap_lengths[gap_lengths > 0]

for zone, df_zone in df_aligned_zones.items():
    print(f"\n📊 Analyse des profils de pannes pour la Zone {zone} :")
    
    for col in ['temp_dry', 'wind_speed', 'wind_dir']:
        gaps = analyze_gap_lengths(df_zone[col])
        
        if len(gaps) > 0:
            max_gap_steps = gaps.max()
            # Conversion en durée lisible (1 step = 10 minutes)
            max_gap_duration = pd.Timedelta(minutes=int(max_gap_steps * 10))
            
            # Calcul des statistiques de pannes
            mean_gap = gaps.mean()
            total_gaps_events = len(gaps)
            
            print(f"  🔹 Feature [{col}] :")
            print(f"    - Nombre d'épisodes de coupure : {total_gaps_events}")
            print(f"    - Durée MOYENNE des coupures  : {mean_gap:.1f} pas ({mean_gap*10:.1f} min)")
            print(f"    - Plus GRAND trou détecté       : {max_gap_steps} pas consécutifs (Soit: {max_gap_duration})")
            
            # Alerte si le trou dépasse 3 heures (18 pas de 10 min)
            if max_gap_steps > 18:
                print(f"    ⚠️ CRITIQUE : Cette feature présente des trous de longue durée. L'interpolation linéaire simple sera proscrite.")
        else:
            print(f"  🔹 Feature [{col}] : 🎉 0 trou détecté sur toute la période !")

⏳ Chargement des données brutes pour audit global...
✅ Dataset chargé : 315490 lignes de relevés élémentaires.

1. AUDIT DE LA GRANULARITÉ TEMPORELLE

[Zone DK1] Fréquence des intervalles de temps détectés :
  - Intervalle : 0 days 00:10:00 -> 157708 fois
  - Intervalle : 0 days 00:20:00 -> 30 fois
  - Intervalle : 0 days 00:40:00 -> 3 fois
  - Intervalle : 0 days 00:30:00 -> 3 fois
  - Intervalle : 0 days 01:10:00 -> 1 fois
  - Intervalle : 0 days 02:50:00 -> 1 fois
  - Intervalle : 0 days 00:50:00 -> 1 fois

[Zone DK2] Fréquence des intervalles de temps détectés :
  - Intervalle : 0 days 00:10:00 -> 157692 fois
  - Intervalle : 0 days 00:20:00 -> 42 fois
  - Intervalle : 0 days 00:30:00 -> 4 fois
  - Intervalle : 0 days 01:20:00 -> 1 fois
  - Intervalle : 0 days 02:50:00 -> 1 fois
  - Intervalle : 0 days 00:50:00 -> 1 fois

2. RECONSTRUCTION ET COMPTAGE DES NaN DIRECTS & SQUELETTIQUES
Plage théorique attendue : 2022-01-01 00:00:00 à 2024-12-31 23:50:00
Nombre d'horodatages (timestamp

In [ ]:
import pandas as pd
import numpy as np
import os

print("⏳ Chargement du dataset complet et valide...")
df_raw = pd.read_csv('raw_data/dmi_weather_raw.csv')
df_raw['HourUTC'] = pd.to_datetime(df_raw['HourUTC'])

# Troncature à l'heure pour préparer l'agrégation
df_raw['HourTime'] = df_raw['HourUTC'].dt.floor('h')

# ==========================================================
# 1. TRAITEMENT VECTORIEL DE LA DIRECTION DU VENT
# ==========================================================
print("🔄 Conversion de wind_dir en composantes vectorielles (U, V)...")
# Correction : on applique directement sur df_raw qui est déjà au format "large"
wind_dir_rad = np.radians(df_raw['wind_dir'])
df_raw['wind_u'] = df_raw['wind_speed'] * np.cos(wind_dir_rad)
df_raw['wind_v'] = df_raw['wind_speed'] * np.sin(wind_dir_rad)

# ==========================================================
# 2. DOWNSAMPLING HORAIRE (Moyenne des blocs de 10 min)
# ==========================================================
print("📉 Agrégation horaire (10-min -> 1h)...")
df_hourly = df_raw.groupby(['HourTime', 'PriceArea', 'Station_ID']).agg({
    'temp_dry': 'mean',
    'wind_speed': 'mean',
    'wind_u': 'mean',
    'wind_v': 'mean'
}).reset_index()

# ==========================================================
# 3. ALIGNEMENT TIMELINE & IMPUTATION VECTORIELLE DES TROUS
# ==========================================================
print("📅 Alignement final sur la timeline du marché spot (2022-2024)...")
start_market = pd.to_datetime('2022-01-01 00:00:00')
end_market = pd.to_datetime('2024-12-31 23:00:00')
market_timeline = pd.date_range(start=start_market, end=end_market, freq='h')

cleaned_datasets = []
features_to_impute = ['temp_dry', 'wind_speed', 'wind_u', 'wind_v']

for zone in ['DK1', 'DK2']:
    df_zone = df_hourly[df_hourly['PriceArea'] == zone].copy()
    
    # Forcer la structure complète du marché horaire
    df_zone = df_zone.set_index('HourTime').reindex(market_timeline).reset_index()
    df_zone = df_zone.rename(columns={'index': 'HourUTC'})
    
    # Restauration des métadonnées structurelles
    df_zone['PriceArea'] = zone
    df_zone['Station_ID'] = '06080' if zone == 'DK1' else '06180'
    
    # Imputation linéaire dans l'espace VECTORIEL
    df_zone[features_to_impute] = df_zone[features_to_impute].interpolate(
        method='linear', limit=6, limit_direction='both'
    )
    
    # Sécurité Production : Suppression stricte des derniers NaNs si panne longue
    df_zone[features_to_impute] = df_zone[features_to_impute].ffill().bfill()
    
    # ==========================================================
    # 4. RECONSTRUCTION DE LA DIRECTION DU VENT POST-IMPUTATION
    # ==========================================================
    df_zone['wind_dir'] = np.degrees(np.arctan2(df_zone['wind_v'], df_zone['wind_u']))
    df_zone['wind_dir'] = (df_zone['wind_dir'] + 360) % 360
    
    # Nettoyage des colonnes vectorielles temporaires
    df_zone = df_zone.drop(columns=['wind_u', 'wind_v'])
    
    cleaned_datasets.append(df_zone)

# Combinaison finale
df_weather_final = pd.concat(cleaned_datasets, ignore_index=True)
df_weather_final = df_weather_final.sort_values(['HourUTC', 'PriceArea']).reset_index(drop=True)

# Sauvegarde
os.makedirs('clean_data', exist_ok=True)
df_weather_final.to_csv('clean_data/dmi_weather_clean.csv', index=False)

print("\n=== FIN DU NETTOYAGE SÉCURISÉ ===")
print(f"Lignes générées          : {df_weather_final.shape[0]} (Attendu : {len(market_timeline) * 2})")
print(f"Valeurs manquantes (NaN) : {df_weather_final.isna().sum().sum()}")
print("🚀 Dataset météo qualifié à 100% pour la production.")

⏳ Chargement du dataset complet et valide...
🔄 Conversion de wind_dir en composantes vectorielles (U, V)...
📉 Agrégation horaire (10-min -> 1h)...
📅 Alignement final sur la timeline du marché spot (2022-2024)...

=== FIN DU NETTOYAGE SÉCURISÉ ===
Lignes générées          : 52608 (Attendu : 52608)
Valeurs manquantes (NaN) : 0
🚀 Dataset météo qualifié à 100% pour la production.
